In [1]:
# RNNによる文章生成
# 言語モデルを使った文章生成
# RNNによる文章生成の手順

In [3]:
# RnnlmGenクラスの実装
import sys
sys.path.append("..")
import numpy as np
from common.functions import softmax
from ch06.rnnlm import Rnnlm
from ch06.better_rnnlm import BetterRnnlm

class RnnlmGen(Rnnlm):
    # state_id：最初にあたえる単語のID
    # skip_ids：生成したくない単語のIDリスト
    # sample_size：全部で何単語分生成するか
    def generate(self, start_id, skip_ids=None, sample_size=100):
        # word_ids：最終的な「生成された文章」を格納するリスト
        # まずは最初の単語（state_id）を入れておく
        word_ids = [start_id]

        # x：今見ている単語
        x = start_id
        while len(word_ids) < sample_size:
            # x（単語ID）を、モデルが受け取れる形にする(バッチサイズ, 時系列)
            x = np.array(x).reshape(1, 1)
            # predict：今の単語xの次の単語の予想スコアを計算する
            score = self.predict(x)
            # softmax：スコアを確率分布に変換
            p = softmax(score.flatten())
            
            # 計算した確率「p」に基づいて、次の単語を1つ選ぶ
            sampled = np.random.choice(len(p), size=1, p= p)
            # もし選んだ単語が「skip_ids」でなければ、
            if (skip_ids is None) or (sampled not in skip_ids):
                # 選んだ単語を、次のループの「今の単語」としてセットする
                x = sampled
                # リストにメモ
                word_ids.append(int(x))

        return word_ids

In [5]:
# 文章生成の実装
import sys
sys.path.append("..")
from rnnlm_gen import RnnlmGen
from dataset import ptb

# corpus：単語IDが並んだリスト
# word_to_id：「単語」から「ID」を引くための辞書
# id_to_word：「ID」から「単語」を引くための辞書
corpus, word_to_id, id_to_word = ptb.load_data("train")
vocab_size = len(word_to_id)  # 何種類の単語が登場するのか
corpus_size = len(corpus)     # 単語の総数

model = RnnlmGen()

# start文字とskip文字の設定
start_word = "you"  # 始めの単語
start_id = word_to_id[start_word]  # 'you'を対応するIDに変換
skip_words = ["N", "<unk>", "$"]   # 生成に使いたくない単語を指定
skip_ids = [word_to_id[w] for w in skip_words]  # skip_idsを単語IDに変換

# 文章生成
# 単語IDで文章を生成
word_ids = model.generate(start_id, skip_ids)
# リストの中の数字をを1つずつ取り出し、辞書で単語に変換
# " ".join(...)：バラバラの単語を半角スペースでつないで、1つの長い文章にする
txt = " ".join([id_to_word[i] for i in word_ids])
# <eos>は文の終わり
# これをピリオドに置き換えて、人間が読みやすい形式に整える
txt = txt.replace(" <eos>", ".\n")
print(txt)

you pepsico licensed contrasts garrison bigger permanently murray colony methods protocol della refcorp soldiers atoms bullet forgotten tells aiming desperately consent dubious tumbling highlight victim ge pricing obstacle assassinations justify biggest rosenthal taped jacobs painful scare 'll jose shots tactics walking tailspin comsat alarm fend grey backgrounds marketers unrelated potato medium-sized raises anticipates dow panel gone fish us$ magna tailspin images ducks trudeau lesser wayne en stunned adds rises ex-dividend refused asarco roberts revealed monday fix merged utsumi guaranty sung aquino cost-cutting sue may eat contended handle wsj fidelity persistent linda considerably father convinced activities brink brooklyn raising porter confrontation


In [6]:
# さらに良い文章へ
# seq2seq
# seq2seqの原理
# 時期列データ変換用のトイ・プロブレム
# 可変長の時系列データ

In [10]:
# 足し算データセット
import sys
sys.path.append("..")
from dataset import sequence

(x_train, t_train), (x_test, t_test) =  \
    sequence.load_data("addition.txt", seed=1984)
char_to_id, id_to_char = sequence.get_vocab()

print(x_train.shape, t_train.shape)
print(x_test.shape, t_test.shape)
print("")
print(x_train[0])
print(t_train[0])
print("")
print("".join([id_to_char[c] for c in x_train[0]]))
print("".join([id_to_char[c] for c in t_train[0]]))

(45000, 7) (45000, 5)
(5000, 7) (5000, 5)

[ 3  0  2  0  0 11  5]
[ 6  0 11  7  5]

71+118 
_189 


In [ ]:
# seq2seqの実装
# Encoderクラス
from common.time_layers import TimeEmbedding
from common.time_layers import TimeLSTM

class Encoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        # V：語彙数
        # D：単語ベクトルの次元数
        # H：隠れ状態の次元数
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        # Embedding層の重み
        embed_W = (rn(V, D) / 100).astype("f")
        # LSTM層の重み
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype("f")
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype("f")
        lstm_b = np.zeros(4 * H).astype("f")
        
        # 時系列データをまとめて処理できるEmbedding層を作成
        self.embed = TimeEmbedding(embed_W)
        # 時系列データを処理する、LSTM層を作成
        # ststeful=False：新しい文章を読み込むたびに記憶をリセット
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)

        # 重みを1つにまとめる
        self.params = self.embed.params + self.lstm.params
        # 勾配を1つにまとめる
        self.grads = self.embed.grads + self.lstm.grads
        # forward処理で、隠れ状態を一時的に保存しておくための箱
        self.hs = None

    def forward(self, xs):
        # 入力された単語IDをベクトルに変換
        xs = self.embed.forward(xs)
        # ベクトル化されたデータをLSTMに入れる
        hs = self.lstm.forward(xs)
        # 計算した隠れ状態を保存する
        self.hs = hs
        # hs野中から最後の時刻だけを抜き出して返す
        return hs[:, -1, :]

    def backward(self, dh):
        # hsと同じ形の0の行列を作成
        dhs = np.zeros_like(self.hs)
        # 
        dhs[:, -1, :] = dh

        dout = self.lstm.backward(dhs)
        dout = self.embed.backward(dout)
        return dout


In [ ]:
# Decoderクラス
